# ___`MR-PMM` using `brms`___
---------------------

In [1]:
print(R.version$version.string)

[1] "R version 4.6.1 (2026-06-24 ucrt)"


In [2]:
# https://cran.r-project.org/web/packages/brms/vignettes/brms_phylogenetics.html

# https://benjamin-halliwell.github.io/MR-PMM/MR-PMM_tutorial.html
# https://benjamin-halliwell.github.io/MR-PMM/MR-PMM_euc_example_analysis.html

# https://paulbuerkner.com/brms/articles/index.html
# https://paulbuerkner.com/brms/articles/brms_multivariate.html
# https://paulbuerkner.com/brms/articles/brms_phylogenetics.html
# https://paulbuerkner.com/brms/reference/brmsformula.html

# https://journal.r-project.org/articles/RJ-2018-017/index.html

In [3]:
suppressPackageStartupMessages({
    library("brms")
    library("rstan")
    library("cmdstanr")
    library("ggplot2")
    library("bayesplot")
    library("loo")
    library("ggpubr")
    library("posterior")
})

In [8]:
tree <- ape::read.tree("../../data/chapter2/uphylomaker/FRED4_1301.tre")
if(!ape::is.binary(tree)) tree <- ape::multi2di(tree)
stopifnot(all(tree$tip.label %in% fred4$binominal))

corrmat <- ape::vcv.phylo(phy = tree, corr = TRUE) # the issue with ape::vcv.phylo is that it uses a BM model to compute the variance covariance matrix - WE WANT TO USE AN OU PROCESS INSTEAD
# look up - https://discourse.mc-stan.org/t/rescaling-phylogenetic-tree-for-non-brownian-phylogenetic-models-in-brms/29637/4
# for this we can use rescale.phylo function from the R library geiger to transform a phylogeny into a OU phylogeny

In [9]:
# mvbind() is used to bind multivariate response variables in brms

# https://paulbuerkner.com/brms/articles/brms_multivariate.html
# if we were to include a categorical trait that has repetitions and we want the model to factor in the within group correlations in that trait across all the response variables,
# we specify that trait the following way 
# (1|p|trait) - here p is just an arbitrary choice - this template means - fit a random intercept over "trait" and consider within group correlations in that trait across all the response variables
# we can choose any alphabet or a number of our liking instead of p in the above formula
# HOWEVER, ONCE SPECIFIED, REUSE OF THAT SYMBOL AGAIN e.g. (1+x|i|g) and (1+z|i|g) WILL LEAD TO CORRELATIONS BEING COMPUTED BETWEEN THE TWO DIFFERENT INSTANCES OF THE GROUP EFFECT i

In [ ]:
# thread on mc-stan discourse regarding model specification
# https://discourse.mc-stan.org/t/multilevel-multiresponse-phylogenetic-mixed-models-with-brms/41411

brms::brm(brms::brmsformula(mvbind(F00727, F00679, F00709) ~ state + (1|p|gr(binominal, cov = corrmat)) + (1|q|taxa)) + set_rescor(TRUE), # state is our predictor (fixed effect)
                   # (1|p|gr(binominal, cov = corrmat)) specifies that the relationships between the `binominal` are to be inferred from the covariance matrix `cov`
                   # and the effects are considred correlated across the response variables - |p|
                   # our fourmula includes two random effects
                   # (1|p|gr(binominal, cov = corrmat)) will capture phylogenetic between species trait correlation while (1|q|taxa) will capture the non-phylogenetic between species trait correlation
                   # (1|q|taxa) is a basic multilevel model with a varying intercept over species (taxa) - with cross response correlations allowed
                   # in (1|p|gr(binominal, cov = corrmat)), by using cov = corrmat in the gr function, we make sure that species are correlated as specified by the covariance matrix corrmat. 
                   data = fred4,
                   data2 = list(corrmat = corrmat), # data2 argument is to pass extra data that does not conform to the formulae's format
                   chains = 6, # each chain will be run in a separate process
                   cores = 6, # set this to the number of chains
                   threads = 4, # within chain (process) parallelization
                   iter = 10000)

# rescor - residual correlation - indicates if residual correlation between the response variables should be modeled.
# currently this is only possible in multivariate gaussian and student models and only relevant in multivariate models

In [ ]:
# control = list(max_treedepth = 15) - this is how you ovveride the default max_treedepth parameter
# https://discourse.mc-stan.org/t/how-many-chains-i-need-for-my-model/8511

In [ ]:
# reporting brms model results in a manuscript
#-----------------------------------------------

# https://www.sciencedirect.com/science/article/pii/S037811272600397X?pes=vor&utm_source=clarivate&getft_integrator=clarivate
# https://www.nature.com/articles/s41559-025-02945-8
# https://www.sciencedirect.com/science/article/pii/S2351989426002398
# https://www.sciencedirect.com/science/article/pii/S0039606026003156
# https://www.science.org/doi/epdf/10.1126/science.adl6282?src=getftr&utm_source=clarivate&getft_integrator=clarivate

# ___Post fit analyses___
-------------------

In [3]:
# https://paulbuerkner.com/brms/articles/brms_multivariate.html
# https://paulbuerkner.com/brms/articles/brms_phylogenetics.html

In [4]:
options(mc.cores = 1)

### ___Models with a fixed effect___
-------------------

In [4]:
# M1 - (RD,SRL,RTD) ~ state + (1│gr(binominal, cov=vcv_phy)) + (1|taxa)
M1 <- readRDS("../../data/chapter2/rdata/hie-general/brms/brms_phylo.Rds")

In [5]:
p1 <- brms::pp_check(M1, resp = "F00679") # RD
p2 <- brms::pp_check(M1, resp = "F00727") # SRL
p3 <- brms::pp_check(M1, resp = "F00709") # RTD

combined <- ggpubr::ggarrange(p1, p2, p3, ncol = 3, labels = c("RD", "SRL", "RTD"))
ggplot2::ggsave("../../plots/pp_check_M1.png", plot = combined, device = "png", width = 30, height = 10, units = "in", dpi = 500, bg = "white")

allpanels <- plot(M1, ask = FALSE, plot = FALSE) # without ask = FALSE, it will prompt you to hit enter to view each one of the 4 plots
p4 <- ggpubr::ggarrange(plotlist = allpanels)
ggplot2::ggsave("../../plots/M1.png", plot = p4, device = "png", width = 32, height = 18, units = "in", dpi = 600, bg = "white")

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.



In [6]:
brms::bayes_R2(M1)

,Estimate,Est.Error,Q2.5,Q97.5
R2F00727,0.4600149,0.009357667,0.4413513,0.4778399
R2F00679,0.4021310,0.010182581,0.3821552,0.4218128
R2F00709,0.6072157,0.006870913,0.5934307,0.6204132


In [7]:
brms::WAIC(M1)

Warning message:
"Ignoring relative efficiencies as some were NA. See argument 'r_eff' in ?loo::loo for more details."
Warning message:
"
962 (18.4%) p_waic estimates greater than 0.4. We recommend trying loo instead."



Computed from 20000 by 5218 log-likelihood matrix.

          Estimate    SE
elpd_waic -13345.8 282.1
p_waic      1618.4  72.4
waic       26691.6 564.1

962 (18.4%) p_waic estimates greater than 0.4. We recommend trying loo instead. 

In [ ]:
# M2 - (RD,SRL,RTD) ~ state + (1|taxa)
M2 <- readRDS("../../data/chapter2/rdata/hie-general/brms/brms_taxa.Rds")

In [10]:
p1 <- brms::pp_check(M2, resp = "F00679")
p2 <- brms::pp_check(M2, resp = "F00727")
p3 <- brms::pp_check(M2, resp = "F00709")

combined <- ggpubr::ggarrange(p1, p2, p3, ncol = 3, labels = c("RD", "SRL", "RTD"))
ggplot2::ggsave("../../plots/pp_check_M2.png", plot = combined, device = "png", width = 30, height = 10, units = "in", dpi = 500, bg = "white")

allpanels <- plot(M2, ask = FALSE, plot = FALSE)
p4 <- ggpubr::ggarrange(plotlist = allpanels)
ggplot2::ggsave("../../plots/M2.png", plot = p4, device = "png", width = 32, height = 18, units = "in", dpi = 600, bg = "white")

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.



In [11]:
brms::bayes_R2(M2)

,Estimate,Est.Error,Q2.5,Q97.5
R2F00727,0.3893356,0.013439667,0.3629723,0.4157702
R2F00679,0.3726282,0.012704067,0.3473084,0.3973292
R2F00709,0.5887281,0.008496843,0.5716138,0.6050943


In [12]:
brms::WAIC(M2)

Warning message:
"Ignoring relative efficiencies as some were NA. See argument 'r_eff' in ?loo::loo for more details."
Warning message:
"
1444 (27.7%) p_waic estimates greater than 0.4. We recommend trying loo instead."



Computed from 20000 by 5218 log-likelihood matrix.

          Estimate    SE
elpd_waic -13883.1 270.9
p_waic      2179.1  76.1
waic       27766.3 541.9

1444 (27.7%) p_waic estimates greater than 0.4. We recommend trying loo instead. 

In [ ]:
# M3 - (RD,SRL,RTD) ~ state + (1│p|gr(binominal, cov=vcv_phy)) + (1|q|taxa)
# M3 <- readRDS("../../data/chapter2/rdata/hie-general/brms/brms_phylo_corr.Rds")

# p1 <- brms::pp_check(M3, resp = "F00679")
# p2 <- brms::pp_check(M3, resp = "F00727")
# p3 <- brms::pp_check(M3, resp = "F00709")

# combined <- ggpubr::ggarrange(p1, p2, p3, ncol = 3, labels = c("RD", "SRL", "RTD"))
# ggplot2::ggsave("../../plots/pp_check_M3.png", plot = combined, device = "png", width = 30, height = 10, units = "in", dpi = 500, bg = "white")

# allpanels <- plot(M3, ask = FALSE, plot = FALSE)
# p4 <- ggpubr::ggarrange(plotlist = allpanels)
# ggplot2::ggsave("../../plots/M3.png", plot = p4, device = "png", width = 32, height = 18, units = "in", dpi = 600, bg = "white")

In [ ]:
# M4 - (RD,SRL,RTD) ~ state + (1|q|taxa)
M4 <- readRDS("../../data/chapter2/rdata/hie-general/brms/brms_taxa_corr.Rds")

In [13]:
p1 <- brms::pp_check(M4, resp = "F00679")
p2 <- brms::pp_check(M4, resp = "F00727")
p3 <- brms::pp_check(M4, resp = "F00709")

combined <- ggpubr::ggarrange(p1, p2, p3, ncol = 3, labels = c("RD", "SRL", "RTD"))
ggplot2::ggsave("../../plots/pp_check_M4.png", plot = combined, device = "png", width = 30, height = 10, units = "in", dpi = 500, bg = "white")

allpanels <- plot(M4, ask = FALSE, plot = FALSE)
p4 <- ggpubr::ggarrange(plotlist = allpanels)
ggplot2::ggsave("../../plots/M4.png", plot = p4, device = "png", width = 32, height = 18, units = "in", dpi = 600, bg = "white")

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.



In [14]:
brms::bayes_R2(M4)

,Estimate,Est.Error,Q2.5,Q97.5
R2F00727,0.5815215,0.007449843,0.5665253,0.5956433
R2F00679,0.5202650,0.008432525,0.5034479,0.5362415
R2F00709,0.6504944,0.006072048,0.6382788,0.6619554


In [15]:
brms::WAIC(M4)

Warning message:
"Ignoring relative efficiencies as some were NA. See argument 'r_eff' in ?loo::loo for more details."
Warning message:
"
1417 (27.2%) p_waic estimates greater than 0.4. We recommend trying loo instead."



Computed from 20000 by 5218 log-likelihood matrix.

          Estimate    SE
elpd_waic -13028.9 303.0
p_waic      1979.5  67.8
waic       26057.7 605.9

1417 (27.2%) p_waic estimates greater than 0.4. We recommend trying loo instead. 

### ___Models with no fixed effect___
---------------------

In [ ]:
# N1 - (RD,SRL,RTD) ~ 1 + (1│gr(binominal, cov=vcv_phy)) + (1|taxa)
N1 <- readRDS("../../data/chapter2/rdata/hie-general/brms/brms_phylo_null.Rds")

In [16]:
p1 <- brms::pp_check(N1, resp = "F00679")
p2 <- brms::pp_check(N1, resp = "F00727")
p3 <- brms::pp_check(N1, resp = "F00709")

combined <- ggpubr::ggarrange(p1, p2, p3, ncol = 3, labels = c("RD", "SRL", "RTD"))
ggplot2::ggsave("../../plots/pp_check_N1.png", plot = combined, device = "png", width = 30, height = 10, units = "in", dpi = 500, bg = "white")

allpanels <- plot(N1, ask = FALSE, plot = FALSE)
p4 <- ggpubr::ggarrange(plotlist = allpanels)
ggplot2::ggsave("../../plots/N1.png", plot = p4, device = "png", width = 32, height = 18, units = "in", dpi = 600, bg = "white")

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.



In [17]:
brms::bayes_R2(N1)

,Estimate,Est.Error,Q2.5,Q97.5
R2F00727,0.4587789,0.009512003,0.4399347,0.4771539
R2F00679,0.4010648,0.010297635,0.3804812,0.4208759
R2F00709,0.6080935,0.006827381,0.5946436,0.6212306


In [18]:
brms::WAIC(N1)

Warning message:
"Ignoring relative efficiencies as some were NA. See argument 'r_eff' in ?loo::loo for more details."
Warning message:
"
978 (18.7%) p_waic estimates greater than 0.4. We recommend trying loo instead."



Computed from 20000 by 5218 log-likelihood matrix.

          Estimate    SE
elpd_waic -13344.1 281.9
p_waic      1634.0  72.9
waic       26688.2 563.7

978 (18.7%) p_waic estimates greater than 0.4. We recommend trying loo instead. 

In [ ]:
# N2 - (RD,SRL,RTD) ~ 1 + (1|taxa)
N2 <- readRDS("../../data/chapter2/rdata/hie-general/brms/brms_taxa_null.Rds")

In [19]:
p1 <- brms::pp_check(N2, resp = "F00679")
p2 <- brms::pp_check(N2, resp = "F00727")
p3 <- brms::pp_check(N2, resp = "F00709")

combined <- ggpubr::ggarrange(p1, p2, p3, ncol = 3, labels = c("RD", "SRL", "RTD"))
ggplot2::ggsave("../../plots/pp_check_N2.png", plot = combined, device = "png", width = 30, height = 10, units = "in", dpi = 500, bg = "white")

allpanels <- plot(N2, ask = FALSE, plot = FALSE)
p4 <- ggpubr::ggarrange(plotlist = allpanels)
ggplot2::ggsave("../../plots/N2.png", plot = p4, device = "png", width = 32, height = 18, units = "in", dpi = 600, bg = "white")

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.



In [20]:
brms::bayes_R2(N2)

,Estimate,Est.Error,Q2.5,Q97.5
R2F00727,0.3933508,0.013640342,0.3664716,0.4198014
R2F00679,0.3747787,0.012855579,0.3494487,0.3994185
R2F00709,0.5916876,0.008396312,0.5748035,0.6078487


In [21]:
brms::WAIC(N2)

Warning message:
"Ignoring relative efficiencies as some were NA. See argument 'r_eff' in ?loo::loo for more details."
Warning message:
"
1454 (27.9%) p_waic estimates greater than 0.4. We recommend trying loo instead."



Computed from 20000 by 5218 log-likelihood matrix.

          Estimate    SE
elpd_waic -13882.2 271.2
p_waic      2232.8  77.0
waic       27764.4 542.3

1454 (27.9%) p_waic estimates greater than 0.4. We recommend trying loo instead. 

In [ ]:
# N3 - (RD,SRL,RTD) ~ 1 + (1│p|gr(binominal, cov=vcv_phy)) + (1|q|taxa)
# N3 <- readRDS("../../data/chapter2/rdata/hie-general/brms/brms_phylo_corr_null.Rds")

# p1 <- brms::pp_check(N3, resp = "F00679")
# p2 <- brms::pp_check(N3, resp = "F00727")
# p3 <- brms::pp_check(N3, resp = "F00709")

# combined <- ggpubr::ggarrange(p1, p2, p3, ncol = 3, labels = c("RD", "SRL", "RTD"))
# ggplot2::ggsave("../../plots/pp_check_N3.png", plot = combined, device = "png", width = 30, height = 10, units = "in", dpi = 500, bg = "white")

# allpanels <- plot(N3, ask = FALSE, plot = FALSE)
# p4 <- ggpubr::ggarrange(plotlist = allpanels)
# ggplot2::ggsave("../../plots/N3.png", plot = p4, device = "png", width = 32, height = 18, units = "in", dpi = 600, bg = "white")

In [ ]:
# N4 - (RD,SRL,RTD) ~ 1 + (1|q|taxa)
N4 <- readRDS("../../data/chapter2/rdata/hie-general/brms/brms_taxa_corr_null.Rds")

In [22]:
p1 <- brms::pp_check(N4, resp = "F00679")
p2 <- brms::pp_check(N4, resp = "F00727")
p3 <- brms::pp_check(N4, resp = "F00709")

combined <- ggpubr::ggarrange(p1, p2, p3, ncol = 3, labels = c("RD", "SRL", "RTD"))
ggplot2::ggsave("../../plots/pp_check_N4.png", plot = combined, device = "png", width = 30, height = 10, units = "in", dpi = 500, bg = "white")

allpanels <- plot(N4, ask = FALSE, plot = FALSE)
p4 <- ggpubr::ggarrange(plotlist = allpanels)
ggplot2::ggsave("../../plots/N4.png", plot = p4, device = "png", width = 32, height = 18, units = "in", dpi = 600, bg = "white")

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.

Using 10 posterior draws for ppc type 'dens_overlay' by default.



In [23]:
brms::bayes_R2(N4)

,Estimate,Est.Error,Q2.5,Q97.5
R2F00727,0.5812559,0.007505432,0.5661522,0.5956524
R2F00679,0.5203098,0.008543080,0.5032474,0.5367349
R2F00709,0.6503505,0.006052755,0.6381690,0.6617760


In [24]:
brms::WAIC(N4)

Warning message:
"Ignoring relative efficiencies as some were NA. See argument 'r_eff' in ?loo::loo for more details."
Warning message:
"
1421 (27.2%) p_waic estimates greater than 0.4. We recommend trying loo instead."



Computed from 20000 by 5218 log-likelihood matrix.

          Estimate    SE
elpd_waic -13037.6 303.5
p_waic      1994.0  68.2
waic       26075.2 607.0

1421 (27.2%) p_waic estimates greater than 0.4. We recommend trying loo instead. 